# 04 — Marker profiles and within-biopsy neighborhoods

KMeans partitions marker-score profiles using a configured k; these are not inferred cell proportions or validated anatomical niches. Neighborhood enrichment and distance co-occurrence are calculated separately in each biopsy. Undefined enrichment Z-scores are kept as missing, not declared significant.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from src import preprocessing, utils, visualization
from src.spatial import build_spatial_graph, validate_graph, spatial_lr
settings = utils.load_settings()
params = settings['params']
seed = params['seed']
sc.settings.set_figure_params(dpi=params['visualization']['figure_dpi'], facecolor='white')

In [ ]:
from sklearn.cluster import KMeans
run = utils.begin_stage(settings, 'adata_profiles.h5ad', 'adata_scored.h5ad')
adata = sc.read_h5ad(run / 'processed' / 'adata_scored.h5ad')
score_cols = list(adata.uns['marker_score_columns'])
values = adata.obs[score_cols].to_numpy()
if not np.isfinite(values).all():
    raise ValueError('Non-finite marker scores')
k = params['marker_profiles']['n_clusters']
labels = KMeans(n_clusters=k, random_state=seed, n_init=10).fit_predict(values)
adata.obs['marker_profile'] = pd.Categorical(labels.astype(str))
utils.profile_summary(adata).to_csv(run / 'tables' / 'marker_profile_frequencies_by_sample.csv', index=False)
adata.obs.groupby('marker_profile', observed=True)[score_cols].mean().to_csv(run / 'tables' / 'marker_profile_centroids.csv')
visualization.plot_spatial_clusters(adata, cluster_key='marker_profile', save_path=run / 'figures' / 'marker_profiles_on_tissue.png')
plt.show()

In [ ]:
graph = validate_graph(adata)
sp = params['spatial']
neighbor_tables = []
cooccurrence_tables = []
graph_rows = []
adata.uns['sample_spatial_stats'] = {}
samples = sorted(adata.obs['sample'].unique())
fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 5), squeeze=False)
for index, sample in enumerate(samples):
    part = adata[adata.obs['sample'] == sample].copy()
    part.obs['leiden'] = part.obs['leiden'].cat.remove_unused_categories()
    validate_graph(part)
    sq.gr.nhood_enrichment(part, cluster_key='leiden', n_perms=sp['n_perms'], seed=seed, n_jobs=1, show_progress_bar=False)
    sq.gr.co_occurrence(part, cluster_key='leiden', interval=sp['distance_bins'])
    categories = part.obs['leiden'].cat.categories.astype(str)
    zscore = part.uns['leiden_nhood_enrichment']['zscore']
    zscore = np.where(np.isfinite(zscore), zscore, np.nan)
    part.uns['leiden_nhood_enrichment']['zscore'] = zscore
    edges = part.uns['leiden_nhood_enrichment']['count']
    neighbor_tables.append(pd.DataFrame({'sample': sample, 'source': np.repeat(categories, len(categories)), 'target': np.tile(categories, len(categories)), 'zscore': zscore.ravel(), 'edge_count': edges.ravel()}))
    co = part.uns['leiden_co_occurrence']
    # Squidpy uses interval edges; output axis corresponds to interval[1:].
    for distance, occurrence in zip(co['interval'][1:], np.moveaxis(co['occ'], -1, 0), strict=True):
        cooccurrence_tables.append(pd.DataFrame({'sample': sample, 'source': np.repeat(categories, len(categories)), 'target': np.tile(categories, len(categories)), 'distance_pixels': distance, 'co_occurrence': occurrence.ravel()}))
    adata.uns['sample_spatial_stats'][str(sample)] = {'clusters': categories.to_numpy(), 'nhood': part.uns['leiden_nhood_enrichment'], 'co_occurrence': co}
    graph_rows.append({'sample': sample, 'n_spots': part.n_obs, 'undirected_edges': part.obsp['spatial_connectivities'].nnz // 2, 'cross_sample_edges': 0})
    sns.heatmap(zscore, mask=~np.isfinite(zscore), xticklabels=categories, yticklabels=categories, center=0, cmap='RdBu_r', ax=axes[0, index])
    axes[0, index].set_title(str(sample))
fig.suptitle('Within-biopsy cluster neighborhood enrichment (permutation Z-score)')
fig.tight_layout()
fig.savefig(run / 'figures' / 'neighborhood_enrichment_by_sample.png')
plt.show()
pd.concat(neighbor_tables).to_csv(run / 'tables' / 'neighborhood_enrichment_by_sample.csv', index=False)
pd.concat(cooccurrence_tables).to_csv(run / 'tables' / 'co_occurrence_by_sample.csv', index=False)
pd.DataFrame(graph_rows).to_csv(run / 'tables' / 'spatial_graph_checks.csv', index=False)
utils.save_checkpoint(adata, run / 'processed' / 'adata_profiles.h5ad')